# 4. Workflows with MoSDeF and signac

Everything so far produced one system at a time. Real studies produce hundreds,
and the hard part stops being the chemistry and starts being bookkeeping. Which
directory held the 30 percent PEO run in water. Did the hexane runs use the
relaxed chain or the raw backmap. Was `screen_3_final_v2` the one with the
bigger box.

signac fixes that by making the parameters the identity of the run. You declare
a **statepoint**, signac hashes it into a job id and a directory, and from then
on the directory and the parameters cannot disagree.

### What you will do here
- Define a parameter space over solvent, solvent ratio, and polymer chemistry
- Mix two off the shelf force fields, OPLS-AA and GAFF, in a single study
- Write one build function that takes a statepoint and returns a typed system
- Run it across every job, writing engine inputs into each job directory
- Query the resulting data space

In [ ]:
import itertools
from pathlib import Path as FilePath

import numpy as np
import signac

import mbuild as mb
import gmso
from gmso import ForceField
from gmso.parameterization import apply
from gmso.formats import write_gro, write_top

from mbuild.path import hard_sphere_random_walk
from mbuild.simulation import HoomdSimulation, ForcesHandler

import logging
from mbuild import mBuildLogger
mBuildLogger().library_logger.setLevel(logging.ERROR)
gmso.gmso_logger.library_logger.setLevel(logging.ERROR)

## The pieces we are stringing together

Nothing here is new. It is notebook 1 through 3 in sequence.

1. `hard_sphere_random_walk` gives a single chain conformation
2. `Path.backmap` turns it into the chemistry named by the statepoint
3. `HoomdSimulation` relaxes the backmapped strain out
4. `mb.fill_box` solvates it
5. `Compound.to_gmso` plus `apply` types the system
6. `write_gro` and `write_top` produce the engine inputs

The statepoint decides which chemistry, which solvent, and how much of it.

## Two force fields, off the shelf

A study is free to use more than one force field. Here PE, PEO, and PP go
through OPLS-AA, and PLA goes through GAFF, which was built as a general purpose
force field for small organic molecules and covers esters well.

Nothing in the build function knows about that choice. The statepoint names the
molecule, a lookup table names the force field, and `apply` takes a dict, so a
polymer and its solvent can be typed from different files in the same call.

Load each force field once and reuse it. GAFF in particular is a large XML, and
reparsing it in every job would dominate the runtime.

In [ ]:
FORCEFIELDS = {
    "oplsaa": ForceField("oplsaa"),
    "gaff": ForceField("files/gaff.xml"),
    "spce": ForceField("files/spce.xml"),
}

# Each chemistry carries its CGsmiles fragment and the force field that types it
CHEMISTRY = {
    "PE": {"fragment": "{#PE=[>]CC[<]}", "forcefield": "oplsaa"},
    "PEO": {"fragment": "{#PEO=[>]COC[<]}", "forcefield": "oplsaa"},
    "PP": {"fragment": "{#PP=[>]CC(C)[<]}", "forcefield": "oplsaa"},
    "PLA": {"fragment": "{#PLA=[>]C(C)C(=O)O[<]}", "forcefield": "gaff"},
}

# Solvents, same idea
SOLVENTS = {
    "water": {"smiles": "O", "forcefield": "spce"},
    "hexane": {"smiles": "CCCCCC", "forcefield": "oplsaa"},
    "methanol": {"smiles": "CO", "forcefield": "oplsaa"},
}

for name, entry in CHEMISTRY.items():
    print(f"{name:<4} -> {entry['forcefield']}")

## One function, from statepoint to written inputs

Write this as a plain function of the statepoint. Everything the run depends on
is an argument, so there is nothing to remember and nothing to set by hand.

In [ ]:
def build_chain(chemistry, n_beads, seed):
    """A single relaxed chain, typed by whichever force field covers it."""
    entry = CHEMISTRY[chemistry]
    forcefield = FORCEFIELDS[entry["forcefield"]]

    path = hard_sphere_random_walk(
        bead_name=chemistry,
        radius=0.225,
        bond_length=0.45,
        termination=n_beads,
        seed=seed,
    )
    chain = path.backmap(entry["fragment"])
    chain.name = "polymer"

    sim = HoomdSimulation(chain, forcefield=forcefield, r_cut=1.2)
    forces = ForcesHandler()   # bonds, angles, and LJ only
    sim.cap_displacement(n_steps=1000, dt=1, max_displacement=1e-3,
                         forces_handler=forces)
    sim.fire(n_steps=500, n_iterations=3, forces_handler=forces)
    return chain

`ForcesHandler()` restricts the minimization to bonds, angles, and LJ. Leaving
electrostatics and torsions out of a minimization is standard, and it keeps the
protocol identical across all four chemistries.

One practical note. GAFF assigns impropers to the PLA ester, and the GMSO to
HOOMD improper conversion currently labels those types inconsistently, so a
minimization with impropers active will raise. The written inputs carry the full
parameter set either way.

In [ ]:
def build_system(statepoint, outdir):
    """Build, solvate, type, and write one statepoint."""
    chain = build_chain(
        statepoint["chemistry"], statepoint["n_beads"], statepoint["seed"]
    )

    polymer_ff = FORCEFIELDS[CHEMISTRY[statepoint["chemistry"]]["forcefield"]]

    solvent_info = SOLVENTS[statepoint["solvent"]]
    solvent = mb.load(solvent_info["smiles"], smiles=True)
    solvent.name = statepoint["solvent"]
    solvent_ff = FORCEFIELDS[solvent_info["forcefield"]]

    n_solvent = int(statepoint["n_solvent"] * statepoint["solvent_ratio"])
    system = mb.fill_box(
        compound=[chain, solvent],
        n_compounds=[statepoint["n_chains"], n_solvent],
        box=[statepoint["box_length"]] * 3,
        seed=statepoint["seed"],
        overlap=0.15,
    )

    topology = system.to_gmso()
    apply(
        top=topology,
        forcefields={
            "polymer": polymer_ff,
            statepoint["solvent"]: solvent_ff,
        },
        identify_connections=True,
        speedup_by_moltag=True,
    )

    outdir = FilePath(outdir)
    # untyped structure straight from mBuild, for reloading into mBuild or GMSO
    system.save(
        str(outdir / "system.mol2"),
        overwrite=True,
        residues=["polymer", statepoint["solvent"]],
    )
    write_gro(topology, str(outdir / "system.gro"))
    write_top(topology, str(outdir / "system.top"))
    return {
        "n_sites": topology.n_sites,
        "n_solvent": n_solvent,
        "polymer_forcefield": CHEMISTRY[statepoint["chemistry"]]["forcefield"],
        "solvent_forcefield": solvent_info["forcefield"],
    }

## Initializing the data space

`signac.init_project` creates the project. `open_job(statepoint).init()` creates
one job directory, named by the hash of the statepoint. Calling it twice with
the same statepoint gives you the same job, which is what makes reruns safe.

The grid below is 4 chemistries by 3 solvents by 2 ratios, so 24 jobs.

In [ ]:
project = signac.init_project(path="polymer_solvation")
print(project.path)

In [ ]:
chemistries = ["PE", "PEO", "PP", "PLA"]
solvents = ["water", "hexane", "methanol"]
ratios = [0.5, 1.0]

for chemistry, solvent, ratio in itertools.product(chemistries, solvents, ratios):
    statepoint = {
        "chemistry": chemistry,
        "solvent": solvent,
        "solvent_ratio": ratio,
        "n_beads": 20,
        "n_chains": 2,
        "n_solvent": 200,
        "box_length": 5.0,
        "seed": 42,
    }
    project.open_job(statepoint).init()

print(len(project), "jobs")

In [ ]:
for job in sorted(project, key=lambda j: (j.sp.chemistry, j.sp.solvent, j.sp.solvent_ratio)):
    print(job.id[:8], job.sp.chemistry, job.sp.solvent, job.sp.solvent_ratio)

## Running the workflow

Loop over the jobs and call the build function. `job.path` is where the outputs
go, and `job.doc` is a small JSON document that travels with the job, which is
the right place for anything you learn while running.

The `if job.doc.get("built")` guard makes the loop restartable. Interrupt it and
rerun, and it picks up where it left off.

In [ ]:
import time

for job in project:
    if job.doc.get("built"):
        continue
    start = time.time()
    result = build_system(job.sp(), job.path)
    job.doc["built"] = True
    job.doc.update(result)
    job.doc["build_seconds"] = round(time.time() - start, 1)
    print(f"{job.sp.chemistry:<4} {job.sp.solvent:<9} ratio={job.sp.solvent_ratio} "
          f"| {result['polymer_forcefield']:>6} + {result['solvent_forcefield']:<7} "
          f"-> {result['n_sites']:>5} sites  ({job.doc['build_seconds']}s)")

Every job directory now holds a complete input set alongside the statepoint that
produced it.

In [ ]:
example = next(iter(project))
!ls {example.path}

In [ ]:
!cat {example.path}/signac_statepoint.json

## Querying the data space

This is where the payoff shows up. Ask questions of the parameter space instead
of remembering directory names.

In [ ]:
for job in project.find_jobs({"solvent": "water"}):
    print(job.sp.chemistry, job.sp.solvent_ratio, job.doc.n_sites)

In [ ]:
for job in project.find_jobs({"chemistry": "PEO", "solvent_ratio": 1.0}):
    print(job.id[:8], job.sp.solvent, job.doc.n_sites)

You can also slice the space by force field, since the build recorded which one
typed each job. Three quarters of the study went through OPLS-AA and the PLA
quarter went through GAFF, all from the same loop.

In [ ]:
from collections import Counter

counts = Counter(job.doc.get("polymer_forcefield") for job in project)
print(counts)

for job in project.find_jobs({"chemistry": "PLA"}):
    print(job.id[:8], job.sp.solvent,
          f"{job.doc.polymer_forcefield} + {job.doc.solvent_forcefield}",
          job.doc.n_sites)

`project.detect_schema()` reports what actually varies across the space, which
is a useful sanity check after a long build.

In [ ]:
print(project.detect_schema())

A summary table pulls the statepoints and the documents together.

In [ ]:
import pandas as pd

rows = [
    {**job.sp(),
     "polymer_ff": job.doc.get("polymer_forcefield"),
     "solvent_ff": job.doc.get("solvent_forcefield"),
     "n_sites": job.doc.get("n_sites"),
     "seconds": job.doc.get("build_seconds"),
     "job_id": job.id[:8]}
    for job in project
]
df = pd.DataFrame(rows)
df = df[["chemistry", "polymer_ff", "solvent", "solvent_ff",
         "solvent_ratio", "n_sites", "seconds", "job_id"]]
df.sort_values(["chemistry", "solvent", "solvent_ratio"]).reset_index(drop=True)

## Extending the space

Adding a parameter is adding a key. Existing jobs keep their ids, new
combinations get new ones, and the guard in the run loop means only the new work
runs.

Below we add a longer chain to the PEO in water runs.

In [ ]:
for ratio in ratios:
    project.open_job({
        "chemistry": "PEO",
        "solvent": "water",
        "solvent_ratio": ratio,
        "n_beads": 40,
        "n_chains": 2,
        "n_solvent": 200,
        "box_length": 5.0,
        "seed": 42,
    }).init()

built = sum(1 for job in project if job.doc.get("built"))
print(len(project), "jobs |", built, "already built")

In [ ]:
for job in project:
    if job.doc.get("built"):
        continue
    result = build_system(job.sp(), job.path)
    job.doc["built"] = True
    job.doc.update(result)
    print(f"new: {job.sp.chemistry} n_beads={job.sp.n_beads} "
          f"ratio={job.sp.solvent_ratio} -> {result['n_sites']} sites")

In [ ]:
for job in sorted(project.find_jobs({"chemistry": "PEO", "solvent": "water"}),
                  key=lambda j: (j.sp.n_beads, j.sp.solvent_ratio)):
    print(f"n_beads={job.sp.n_beads:<3} ratio={job.sp.solvent_ratio} "
          f"-> {job.doc.n_sites} sites")

<h1 style="color: green;">Exercise</h1>

Extend the study.

1. Add a fourth solvent to `SOLVENTS`, for example acetone (`CC(=O)C`) or
   toluene (`Cc1ccccc1`), typed with OPLS-AA
2. Initialize the new jobs and rerun the build loop
3. Query for the jobs that used your new solvent and print their site counts

Then try a second axis. Add `n_chains` to the grid with values 1 and 4, and
check that the box is still big enough by looking at `job.doc["n_sites"]`.

Tip. `signac-flow` is the companion package that turns the run loop into
declarative operations with dependencies, so a real study can be submitted to a
cluster. It is not installed here, but everything above maps onto it directly.

In [ ]:
# Your code here

<h2 style="color: blue;">Answer</h2>

Run the cell below to see one solution.

In [ ]:
SOLVENTS["toluene"] = {"smiles": "Cc1ccccc1", "forcefield": "oplsaa"}

for chemistry, ratio in itertools.product(chemistries, ratios):
    project.open_job({
        "chemistry": chemistry,
        "solvent": "toluene",
        "solvent_ratio": ratio,
        "n_beads": 20,
        "n_chains": 2,
        "n_solvent": 200,
        "box_length": 5.0,
        "seed": 42,
    }).init()

for job in project:
    if job.doc.get("built"):
        continue
    result = build_system(job.sp(), job.path)
    job.doc["built"] = True
    job.doc.update(result)

for job in project.find_jobs({"solvent": "toluene"}):
    print(job.sp.chemistry, job.sp.solvent_ratio, job.doc.n_sites)

---

### Recap

- A statepoint is the identity of a run, and signac makes the directory follow from it
- A lookup keyed by chemistry picks the force field, so OPLS-AA and GAFF coexist
  in one study without any special casing in the build function
- One function of the statepoint replaces a folder of edited input files
- `job.doc` carries the results of the build alongside the parameters
- `find_jobs` and `detect_schema` let you ask questions of the space
- Adding a parameter adds jobs without disturbing the ones already built

That closes the loop. Build with mBuild, type with GMSO and foyer, write for any
engine, and let signac keep the whole study straight.